# [12-3강] GPU Memory 기초 - 실습

In [1]:
import torch
torch.set_num_threads(1)
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import random

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

torch.set_printoptions(precision=4, sci_mode=False)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)


device: cpu


## 문제 1. Tensor 메모리 사용량 계산 함수 만들기

Tensor의 원소 수와 원소당 byte 수로 대략적인 메모리 사용량을 계산합니다.

In [3]:
def tensor_mb(tensor):
    # TODO: tensor.numel()과 tensor.element_size()를 사용해 MB를 계산하세요.
    return tensor.numel() * tensor.element_size() / (1024 ** 2)

x = torch.randn(32, 3, 64, 64)
print('x memory MB:', round(tensor_mb(x), 4))


x memory MB: 1.5


In [2]:
####
def tensor_mb(tensor):
    return tensor.numel() * tensor.element_size() / (1024 ** 2)

x = torch.randn(32, 3, 64, 64)
print('x memory MB:', round(tensor_mb(x), 4))

x memory MB: 1.5


## 문제 2. batch size별 입력 메모리 표 만들기

같은 이미지 크기에서 batch size만 바꾸며 입력 Tensor 메모리가 어떻게 바뀌는지 확인합니다.

In [4]:
def tensor_mb(tensor):
    return tensor.numel() * tensor.element_size() / (1024 ** 2)

batch_sizes = [8, 16, 32, 64]
rows = []
for bs in batch_sizes:
    # TODO: [bs, 3, 64, 64] Tensor를 만들고 메모리를 계산하세요.
    x = torch.randn(bs, 3, 64, 64)
    rows.append((bs, round(tensor_mb(x), 4)))
print(rows)


[(8, 0.375), (16, 0.75), (32, 1.5), (64, 3.0)]


### 해설 및 실행 결과 해석

- batch size가 2배가 되면 입력 Tensor 메모리도 2배가 됩니다. 실제 학습에서는 activation과 gradient도 함께 저장되므로 체감 메모리는 더 커집니다.

## 문제 3. CPU/GPU 공통 메모리 리포트 함수 만들기

GPU가 없을 때도 실행되는 메모리 리포트 함수를 작성합니다.

In [6]:
def memory_report(x):
    # TODO: input_mb와 cuda_allocated_mb를 계산하세요.
    input_mb = x.numel() * x.element_size() / (1024 ** 2)
    if torch.cuda.is_available():
        cuda_allocated_mb = torch.cuda.memory_allocated() / (1024 ** 2)
    else:
      cuda_allocated_mb = 0.0
      return {'input_mb': round(input_mb, 4), 'cuda_allocated_mb': round(cuda_allocated_mb, 4)}

x = torch.randn(16, 3, 32, 32).to(device)
print(memory_report(x))


{'input_mb': 0.1875, 'cuda_allocated_mb': 0.0}


In [5]:
#####
def memory_report(x):
    input_mb = x.numel() * x.element_size() / (1024 ** 2)
    if torch.cuda.is_available():
        cuda_allocated_mb = torch.cuda.memory_allocated() / (1024 ** 2)
    else:
        cuda_allocated_mb = 0.0
    return {'input_mb': round(input_mb, 4), 'cuda_allocated_mb': round(cuda_allocated_mb, 4)}

x = torch.randn(16, 3, 32, 32).to(device)
print(memory_report(x))

{'input_mb': 0.1875, 'cuda_allocated_mb': 0.0}


### 해설 및 실행 결과 해석

- CPU 환경에서는 CUDA memory가 0으로 표시됩니다. GPU 환경에서는 현재 할당된 CUDA memory를 함께 볼 수 있어 batch size 조절 판단에 도움을 줍니다.